In [1]:
from generate_utils import load_GraphModel, load_BiLSTMModel, load_AttnFiLMSEModel, generate_files_with_nucleus
import torch
import numpy as np
import pickle
from tqdm import tqdm
from GridMLM_tokenizers import CSGridMLMTokenizer
from graph_utils import get_graph_embeddings_from_string_with_model, get_bilstm_embeddings_from_string_with_model, graph_from_string, compare_heterodata

/home/maximos/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [3]:
device_name = 'cuda:0'
device = torch.device(device_name)

graph_model_path = 'saved_models/graph/graph_model.pt'
transformer_graph_path = 'saved_models/graph/transformer_model.pt'

bilstm_model_path = 'saved_models/bilstm/bilstm_model.pt'
transformer_bilstm_path = 'saved_models/bilstm/transformer_model.pt'

In [4]:
graph_model = load_GraphModel(graph_model_path, device)
transformer_graph = load_AttnFiLMSEModel(
    tokenizer,
    device,
    checkpoint_path=transformer_graph_path
)

bilstm_model = load_BiLSTMModel(bilstm_model_path, device)
transformer_bilstm = load_AttnFiLMSEModel(
    tokenizer,
    device,
    checkpoint_path=transformer_bilstm_path
)

In [5]:
graph_model.eval()
transformer_graph.eval()
bilstm_model.eval()
transformer_bilstm.eval()

AttnFiLMSEModel(
  (melody_proj): Linear(in_features=13, out_features=512, bias=True)
  (harmony_embedding): Embedding(355, 512)
  (layers): ModuleList(
    (0-7): 8 x TransformerBlockWithAttnFiLM(
      (attn): MultiHeadAttentionWithAttnFiLM(
        (q_proj): Linear(in_features=512, out_features=512, bias=True)
        (k_proj): Linear(in_features=512, out_features=512, bias=True)
        (v_proj): Linear(in_features=512, out_features=512, bias=True)
        (out_proj): Linear(in_features=512, out_features=512, bias=True)
        (q_films): ModuleList(
          (0-7): 8 x IdCenteredFiLM(
            (gamma_proj): Linear(in_features=128, out_features=64, bias=True)
            (beta_proj): Linear(in_features=128, out_features=64, bias=True)
          )
        )
        (k_films): ModuleList(
          (0-7): 8 x IdCenteredFiLM(
            (gamma_proj): Linear(in_features=128, out_features=64, bias=True)
            (beta_proj): Linear(in_features=128, out_features=64, bias=True)
  

In [6]:
datasets = {
    'gjt': {'path': 'data/gjt_test.pkl', 'dataset': None},
    # 'hook': {'path': 'data/hook_test.pkl', 'dataset': None},
    'nott': {'path': 'data/nott_test.pkl', 'dataset': None},
    # 'wiki': {'path': 'data/wiki_test.pkl', 'dataset': None}
}

for k, v in datasets.items():
    print(f'loading {k}')
    with open(v['path'], 'rb') as f:
        d = pickle.load(f)
    v['dataset'] = d

loading gjt
loading nott


In [7]:
# in_seq = 'b_A#:7_@2;A:min6_@2'
# in_seq = 'b_F#:7_@2;B:7_@2b_E:7_@2;A#:7_@2;'
# in_seq = 'b_A#:7_@2;A:min6_@2'
# in_seq = 'b_F:min_@2;A#:min_@2'
in_seq = 'E:min;A:maj;F:min;A#:maj'

In [8]:
y_graph = get_graph_embeddings_from_string_with_model(in_seq, graph_model)
y_bilstm = get_bilstm_embeddings_from_string_with_model(in_seq, bilstm_model)

E:min in vocab as: 124
A:maj in vocab as: 268
F:min in vocab as: 153
A#:maj in vocab as: 297
E:min in vocab as: 124
A:maj in vocab as: 268
F:min in vocab as: 153
A#:maj in vocab as: 297


In [9]:
print(y_graph.unsqueeze(0).shape)
print(y_bilstm.shape)

torch.Size([1, 128])
torch.Size([1, 128])


In [10]:
# input_f_path = '/mnt/ssd2/maximos/data/gjt_melodies/gjt_CA_test/Autumn_Leaves.mxl'
input_f_path = '/mnt/ssd2/maximos/data/gjt_melodies/gjt_CA_test/So_What.mxl'

piece_name = input_f_path.split('/')[-1].split('.')[0]
print(piece_name)

So_What


In [11]:
gen_out = generate_files_with_nucleus(
    transformer_bilstm,
    tokenizer,
    input_f_path=input_f_path,
    mxl_folder_out=None,
    midi_folder_out='midi_tests',
    name_suffix=f'{piece_name}_no',
    guidance_vec = None,
    use_constraints=False,
    intertwine_bar_info=True,
    normalize_tonality=True,
    temperature=1.0,
    p=0.9,
    unmasking_order='certain',
    create_gen=True,
    create_real=False
)
print(gen_out['gen_output_tokens'])

['<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'A:min', 'A:min', 'A:min', 'A:min', '<bar>', 'A:maj', 'A:maj', 'A:maj', 'A:maj', '<bar>', 'F:min', 'F:min', 'F:min', 'F:min', '<bar>', 'D#:maj', 'D#:maj', 'D#:maj', 'D#:maj', '<bar>', 'F:min', 'F:min', 'F:min', 'F:min', '<bar>', 'D#:maj', 'D#:maj', 'F:maj', 'F:maj', '<bar>', 'F:min', 'F:min', 'F:min', 'F:min', '<bar>', 'D#:maj', 'D#:maj', 'D#:maj', 'D#:maj', '<bar>', 'F:maj', 'F:maj', 'F:maj', 'F:maj', '<bar>', 'D#:maj', 'D#:maj', 'F:maj', 'F:maj']


In [12]:
gen_out = generate_files_with_nucleus(
    transformer_bilstm,
    tokenizer,
    input_f_path=input_f_path,
    mxl_folder_out=None,
    midi_folder_out='midi_tests',
    name_suffix=f'{piece_name}_bilstm',
    guidance_vec = y_bilstm,
    use_constraints=False,
    intertwine_bar_info=True,
    normalize_tonality=True,
    temperature=1.0,
    p=0.9,
    unmasking_order='certain',
    create_gen=True,
    create_real=False
)
print(gen_out['gen_output_tokens'])

['<bar>', 'F#:dim7', 'F#:dim7', 'E:min', 'E:min', '<bar>', 'E:min', 'E:min', 'D:maj', 'D:maj', '<bar>', 'E:min', 'E:min', 'E:min', 'A:9', '<bar>', 'E:min', 'E:min', 'E:min', 'E:min', '<bar>', 'F#:dim7', 'F#:dim7', 'E:min7', 'E:min7', '<bar>', 'D:maj', 'D:maj', 'D:maj', 'D:maj', '<bar>', 'F:maj', 'F:maj', 'F:maj', 'F:maj', '<bar>', 'A:min7', 'A:min7', 'C:maj', 'F:sus2', '<bar>', 'C:min', 'C:min', 'C:min', 'C:min', '<bar>', 'D#:maj7', 'C#:min6', 'F:7', 'F:7', '<bar>', 'D#:11', 'D#:11', 'F:min7', 'D#:11', '<bar>', 'F:min11', 'G#:dim7', 'F:7', 'F:7', '<bar>', 'F:min', 'F:min', 'F:min', 'F:min', '<bar>', 'A#:min', 'A#:min', 'F:min11', 'G:7', '<bar>', 'C:min', 'C:min', 'C:min', 'C:min', '<bar>', 'G#:dim7', 'G#:dim7', 'G#:dim7', 'G#:dim7']


In [13]:
gen_out = generate_files_with_nucleus(
    transformer_graph,
    tokenizer,
    input_f_path=input_f_path,
    mxl_folder_out=None,
    midi_folder_out='midi_tests',
    name_suffix=f'{piece_name}_graph',
    guidance_vec = y_graph.unsqueeze(0),
    use_constraints=False,
    intertwine_bar_info=True,
    normalize_tonality=True,
    temperature=1.0,
    p=0.9,
    unmasking_order='certain',
    create_gen=True,
    create_real=False
)
print(gen_out['gen_output_tokens'])

['<bar>', 'E:min9', 'E:min9', 'E:min9', 'E:min9', '<bar>', 'G:min9', 'G:min9', 'G:min9', 'G:min9', '<bar>', 'E:min9', 'E:min9', 'E:min9', 'E:min9', '<bar>', 'F#:min7', 'F#:min7', 'F#:min7', 'F#:min7', '<bar>', 'E:min9', 'E:min9', 'E:min9', 'E:min9', '<bar>', 'G:min9', 'G:min9', 'G:min9', 'G:min9', '<bar>', 'E:min9', 'E:min9', 'E:min9', 'E:min9', '<bar>', 'E:min7', 'E:min7', 'E:min7', 'E:min7', '<bar>', 'F:min', 'F:min', 'F:min', 'F:min', '<bar>', 'C:min7', 'C:min7', 'C:min7', 'C:min7', '<bar>', 'F:min', 'F:min', 'F:min', 'F:min', '<bar>', 'C:min7', 'C:min7', 'C:min7', 'C:min7', '<bar>', 'F:min', 'F:min', 'F:min', 'F:min', '<bar>', 'C:min7', 'C:min7', 'C:min7', 'C:min7', '<bar>', 'F:min', 'F:min', 'F:min', 'F:min', '<bar>', 'G:sus2', 'D#:maj', 'G:sus2', 'G:sus2']
